# LLM graph iteration demo

Anton Antonov   
January 2026

---

## Introduction

This notebook shows how to do iterative refinements of an LLM-based (AI) agent results using a iterative evaluation of an (acyclic) LLM-graph.

In [15]:
[o]({$_ + 1} xx 3)(1)

4

In [17]:
[\o]({$_ + 1} xx 3).map({ $_(1) })

(2 3 4)

In [16]:
[\o]({$_ + 1} xx 3)
andthen $_.map({ $_(1) })

(2 3 4)

In [11]:
[\+] 1..4

(1 3 6 10)

In [7]:
([\o]({$_ + 1} xx 3)).map(*.(1))

No such method 'CALL-ME' for invocant of type 'Whatever'

---

## Setup

In [24]:
use LLM::Graph;
use WWW::Ollama;

In [25]:
sink my $conf-ollama = LLM::Functions::Configuration.new(
        name => 'ollama',
        module => 'WWW::Ollama',
        base-url => ollama-base-url,
        model => 'gemma3:1b',
        function => &ollama-chat-completion,
        embedding-model => 'nomic-embed-text',
        embedding-function => &ollama-embedding,
        temperature => 0.55,
        max-tokens => 8192,
        format => 'values');

In [26]:
sink my $conf41 = llm-configuration('ChatGPT', model => 'gpt-4.1', temperature => 0.55, max-tokens => 8192);

In [27]:
sink my $conf = $conf-ollama;

----

## Graph nodes

The code below implements the following workflow:

1. Generate the first draft

2. Determine whether the draft should criticized 

    - LLM can be used, or (deterministic) lookup of a verdict word or phrase.

3. If, yes, critique the draft

    - The critic gives a verdict "BAD", "ACCEPTABLE", and "GOOD" _and_ criticism notes.

4. If the verdict is not "GOOD" generate a new revision using criticism notes

5. The iterations stops when the critique gives the verdict "GOOD" or max iterations is reached.

### Generation rules

In [28]:
sink my %generation-rules =
    generate => sub ($topic) { "Generate a 200 words thesis text for the topic $topic." ~ llm-prompt('NothingElse')('thesis text without comments')} 
;

### Critique LLM-function

In [36]:
my &cFunc = llm-function( -> $text {
    [
        "Write a VERY SHORT critique of the thesis below.",
        "Give your verdict to be one of 'BAD', 'ACCEPTABLE', 'GOOD'.",
        "If you verdict is 'BAD' or 'ACCEPTABLE' give improvement directions only.",
        "THESIS:\n", 
        $text ~~ Str:D ?? $text !! $text<revision>,
        "THESIS-END\n",
        'Give critique as a JSON dictionary of the form: { "verdict": ..., "comments" : [ ...] }',
        llm-prompt('NothingElse')('JSON')
    ].join("\n")
}, e => $conf, form => sub-parser('JSON'):drop);

LLM::Function(-> **@args, *%args { #`(Block|5405758536528) ... }, 'ollama')

Critique prompt variant:

```raku
    [
        "Write a VERY SHORT critique of the thesis below.",
        "Evaluate with questions like:",
        "1) Is it too short? 2) Does it have at least two sections? 3) Is it easy to read?",
        "Give your verdict to be one of 'BAD', 'ACCEPTABLE', 'GOOD'.\n",
        "THESIS:\n", 
    ]
```    

### Revision LLM-function

In [30]:
my &rFunc = llm-function(-> $critique-form {
    [
        "Here is a thesis:",
        "\nTHESIS:\n", 
        $critique-form<revision>,
        "\nTHESIS-END\n",
        "And here is a critique of it:",
        "CRITIQUE:\n",
        $critique-form<critique>,
        "END-CRITIQUE\n",
        "Improve the thesis text based on the critique.",
        "Do not comment, annotate, or give options"
    ].join("\n")
}, e => $conf);

LLM::Function(-> **@args, *%args { #`(Block|5405919042032) ... }, 'ollama')

### Refinement rules

In [31]:
my %empty-form = critique => '', critique-iteration => 0, revision => '', revision-iteration => 0;

sink my %revision-rules =

    decide => { 
        eval-function => sub ($text) { 
            $text ~~ Str:D || !($text ~~ Map:D && $text<critique>.contains('GOOD'))
        } 
    },

    critique => {
        llm-function => sub ($text) { &cFunc($text) },
        
        test-function => sub ($decide) { $decide.raku.lc.contains('true') }
    },

    critique-form => {
        eval-function => sub ($text, $critique) {
            my $form = $text ~~ Map:D ?? $text.clone !! merge-hash(%empty-form , {revision => $text});
            $form<critique> = $critique;
            $form<critique-iteration> += 1;
            return $form;
        },
        
        test-function => sub ($critique) { $critique.defined }
    },

    revise => {
        llm-function => sub ($critique-form) { &rFunc($critique-form) },
        
        test-function => sub ($critique-form) { $critique-form.defined },
    },

    revise-form => {
        eval-function => sub ($critique-form, $revise) {
            my %form = $critique-form.clone; 
            %form<revision> = $revise;
            %form<revision-iteration> += 1;
            return %form;
        },
        
        test-function => sub ($revise) { $revise.defined }
    },

    finalize => {
        eval-function => sub ($text, $revise-form) { $revise-form.defined ?? $revise-form !! $text}
    }  
;

---


## Build the graphs

In [32]:
my $g1 = LLM::Graph.new(%generation-rules, e => $conf):async;
my $g2 = LLM::Graph.new(%revision-rules, e => $conf):async;

(:$g1, :$g2)

(g1 => LLM::Graph(size => 1, nodes => generate) g2 => LLM::Graph(size => 6, nodes => critique, critique-form, decide, finalize, revise, revise-form))

---

## Run and inspect state transitions

In [33]:
$g1.eval({ topic => "why cyclic graphs help with iterative writing" })

LLM::Graph(size => 1, nodes => generate)

In [34]:
my $text = $g1.nodes<generate><result>

Cyclic graphs, particularly those with a defined number of nodes and edges, offer a powerful framework for iterative writing due to their inherent organizational properties. Their structure naturally facilitates the progression of ideas through distinct stages – exposition, development, and conclusion – mirroring the iterative nature of creative processes. The repetition of patterns and relationships within the graph encourages consistent thematic exploration, subtly guiding the reader through a coherent narrative. The constraint of completing a cycle promotes mindful drafting, minimizing meandering and encouraging focused narrative construction. Furthermore, the visual representation of interconnectedness highlights the dynamic relationship between different elements, stimulating revision and refinement.  Ultimately, the cyclic graph’s design facilitates a streamlined and engaging iterative workflow, enhancing the author’s control and coherence.

| Stage       | Description           

In [12]:
$text.words.elems

194

In [38]:
#% html
&cFunc($text)
==> to-html

verdict,ACCEPTABLE
comments,"The thesis is generally well-structured and articulate. It effectively argues for the cyclical graph’s suitability for iterative writing. However, the language could be more concise and impactful. The connections to ‘creative processes’ are slightly vague; consider explicitly stating *how* the graph facilitates this.The visual representation table is adequate but could be streamlined. Adding a brief explanation of each visual element could enhance clarity.Expand on the 'dynamic relationship between different elements' – provide a concrete example demonstrating this dynamic interaction to strengthen the argument.The thesis could benefit from a stronger concluding statement, emphasizing the overall benefits of using the cyclic graph method."


In [13]:
$g2.eval({ :$text })

LLM::Graph(size => 6, nodes => critique, critique-form, decide, finalize, revise, revise-form)

In [14]:
#% markdown
$g2.nodes<finalize><result>

{critique => Okay, here's a critique of the thesis, followed by a verdict and suggested improvements:

**Critique:**

The thesis is generally well-written and presents a coherent argument. It successfully highlights the benefits of cyclic graphs for iterative writing. However, it leans *slightly* heavily on the "optimized" narrative aspect and could be more explicitly focused on *how* this optimization works. The language is a bit dense and could benefit from more concise phrasing. It also relies on a somewhat broad, generalized claim ("optimizes the writing process").

**Verdict:** ACCEPTABLE

**Improvements:**

1.  **Specificity:**  Instead of just saying "minimizes cognitive load," *specifically* explain *how* the graph reduces it.  For example, "By minimizing cognitive load, the graph encourages readers to actively process information, rather than getting bogged down in complex storylines."

2.  **Focus on the 'Why':** The thesis states the benefit – optimized narrative flow – but doesn’t *explain* why this leads to a more satisfying narrative.  Consider adding a sentence hinting at the emotional or thematic impact of the cycle.  Something like, "This cyclical structure subtly strengthens thematic connections and anticipation, fostering a deeper emotional investment in the narrative."

3.  **Show, Don't Just Tell:** Instead of saying "subtly reinforces thematic connections," provide a *brief* example of how the graph does so. (e.g., “The repeated motif of the abandoned lighthouse subtly highlights themes of loss and memory.")

4. **Strengthen the connection to the core idea:** The thesis says the graph *optimizes* the process.  Hint at the central *goal* of the writing – to create a satisfying narrative.

**Revised Thesis (Example incorporating some of these suggestions):**

"Cyclic graphs offer a strategic advantage for iterative writing by fostering a predictable and manageable structure, effectively reducing the cognitive load associated with navigating complex storylines. The repetition of patterns, loops, and key elements within the graph facilitates a consistent flow of ideas, encouraging readers to actively process information, thereby subtly strengthening thematic connections and anticipating future developments. This cyclical structure subtly strengthens thematic connections and anticipation, fostering a deeper emotional investment in the narrative.”

**Deliverable Verdict:**

**ACCEPTABLE**, critique-iteration => 1, revision => Okay, here’s a revised thesis text, incorporating the critique and incorporating the suggested improvements.

“Cyclic graphs, with their inherent repetition, offer a strategic advantage for iterative writing by fostering a predictable and manageable structure. The repetition of patterns, loops, and key elements within the graph facilitates a consistent flow of ideas, encouraging readers to follow a predetermined path. This structured approach minimizes the cognitive load associated with navigating complex storylines, allowing writers to concentrate on the core narrative without being overwhelmed by repetition. The graph’s design subtly guides the reader, subtly reinforcing thematic connections and anticipating future developments. Its visual nature enhances comprehension and strengthens the sense of agency, promoting a cyclical engagement with the text. Ultimately, the cyclical nature of a graph optimizes the writing process, leading to a more cohesive and satisfying narrative.”

“By reducing cognitive load, the graph encourages readers to actively process information, thereby subtly strengthening thematic connections and anticipating future developments. This cyclical structure subtly strengthens thematic connections and anticipation, fostering a deeper emotional investment in the narrative.”
, revision-iteration => 1}

In [ ]:
$g1.clear;
$g2.clear;

$g1.eval({ topic => "Why cyclic graphs help with iterative writing?" });
my $text = $g1.nodes<generate><result>;

for (^4) -> $iter {
  say '=' x 10, $iter, '=' x 10;
  
  $g2.eval({:$text});  
  
  my $revision = $g2.nodes<finalize><result>;
  last if $revision eq $text;
  $text = $revision;

  say 'critique => ', $g2.nodes<critique><result>;
  say "-=" x 10, "-";
  say to-json($text);
}

==========0==========
critique => Okay, here’s a critique and verdict:

**Critique:**

The thesis is generally well-written and presents a solid argument for the value of cyclic graphs in writing. However, it’s somewhat *too* broad and could benefit from a slightly sharper focus. The language is a little verbose, and it leans heavily on descriptive phrases ("seamless backtracking," "systematic examination"). It also feels a bit generalized – it mentions "narrative threads" but doesn't explore *how* the graph facilitates this.

**Verdict:**

**ACCEPTABLE**

**Improvement Directions:**

1.  **Specificity:** Instead of just saying "narrative threads," briefly mention *what* kind of narrative threads. Are we talking about plot, character arcs, or thematic elements?
2.  **Focus on a Specific Benefit:** Instead of *all* the benefits, pick *one* particularly compelling reason why cyclic graphs are valuable.  For example, you could argue that they're particularly effective for complex stories 

### Final output

In [16]:
#% markdown
$text<revision>

Okay, here’s the revised thesis, incorporating the critique’s feedback:

**Revised Thesis:**

Cyclic graphs offer a powerful framework for iterative writing, particularly effective for complex narratives with multiple branching plot points, such as fantasy or mystery stories. The circular nature of the graph allows writers to strategically revisit earlier choices and refine narrative threads, facilitating consistent exploration of potential paths and promoting a deeper engagement with the text – this repeated exploration allows for a more complete understanding of character motivations and the evolution of the plot.

In [17]:
#% markdown
$g2.nodes<critique-form>

{eval-function => sub { }, input => [critique text], result => (Any), spec-type => (), test-function => sub { }, test-function-input => [critique], test-function-result => False}

----

## LLM graph plots

In [18]:
#% html
my %opts = engine => 'dot', :9graph-size, node-width => 1.2, theme => 'default';
[
    generation-graph => $g1.dot(|%opts, graph-size => 1.5, :svg),
    revision-graph => $g2.dot(|%opts, :5graph-size, :svg)
]
==> to-html-table()

generation-graph revision-graph <?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
 
 generate_Cluster 
 
 
<!-- generate -->
 
 generate 
 
 generate 
 
<!-- topic -->
 
 topic 
 
 topic 
 
<!-- topic->generate -->
 
 topic->generate 
 
 
 
 
 
 <?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- critique -->
 
 critique 
 
 critique 
 
<!-- critique-form -->
 
 critique-form 
 
 critique-form 
 
<!-- critique->critique-form -->
 
 critique->critique-form 
 
 
 
<!-- revise -->
 
 revise 
 
 revise 
 
<!-- critique-form->revise -->
 
 critique-form->revise 
 
 
 
<!-- revise-form -->
 
 revise-form 
 
 revise-form 
 
<!-- critique-form->revise-form -->
 
 critique-form->revise-form 
 
 
 
<!-- decide -->
 
 decide 
 
 decide 
 
<!-- decide->critique -->
 
 decide->critique 
 
 
 
<!-- finalize -->
 
 finalize 
 
 finalize 
 
<!-- revise->revise-form -->
 
 revise->revise-form 
 
 
 
<!-- revise-form->finalize -->
 
 revise-form->finalize 
 
 
 
<!-- text -->
 
 text 
 
 text 
 
<!-- text->critique -->
 
 text->critique 
 
 
 
<!-- text->critique-form -->
 
 text->critique-form 
 
 
 
<!-- text->decide -->
 
 text->decide 
 
 
 
<!-- text->finalize -->
 
 text->finalize